# 01-phase1-reproducibility

Phase 1 (depth-only growing Transformer) 의 **재현성 검증**.

- **검증할 가설**:
  1. **시드 invariance**: 여러 시드 (42 / 123 / 456) 에서 3 핵심 가설 (PlateauTrigger fire / no spike / 깊어지고 낮은 loss) 이 일관되게 통과
  2. **결정론**: 동일 시드 2회 실행 시 동일 grow events + (CPU 의 경우) 동일 loss curve
  3. **분산**: 시드 간 final loss / grow event step 의 표준편차가 합리적 범위
- **데이터**: TinyShakespeare (char-LM)
- **작성일**: 2026-05-24
- **연관**: Issue [#31](https://github.com/EinSofINTEREST/GraphLM/issues/31) / Base 검증 노트북 [00-phase1-growing-decoder.ipynb](00-phase1-growing-decoder.ipynb)

> Base 노트북 (10-phase1) 은 seed=42 단일 실행만 보고. 본 노트북은 시드 다양성과 결정론을 추가 검증.

> ⚠️ **GPU 결정론**: cuBLAS / cuDNN 의 non-deterministic 커널 때문에 GPU 학습은 완전한 비트 단위 결정론을 보장하지 않습니다. 본 노트북의 결정론 검증은 `torch.allclose(rtol=1e-3)` 수준의 근사 결정론을 본다.

## 1. 환경 / 시드 / device

In [ ]:
import logging
import sys

import torch

import graphlm
from graphlm.data.tinyshakespeare import (
    CharTokenizer,
    TinyShakespeareDataset,
    iter_random_batches,
    load_tinyshakespeare_text,
)
from graphlm.models.backbone import BackboneConfig, GrowingDecoder
from graphlm.training.loop import TrainConfig, train
from graphlm.utils import repo_root, set_seed

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

logging.basicConfig(
    level=logging.WARNING, format="%(asctime)s %(levelname)s %(message)s", datefmt="%H:%M:%S"
)

print("python  :", sys.version.split()[0])
print("graphlm :", graphlm.__version__)
print("torch   :", torch.__version__)
print("device  :", DEVICE)
if str(DEVICE).startswith("cuda"):
    print(f"  device 0      : {torch.cuda.get_device_name(0)}")
    print(f"  total VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 2. 실험 설정

Base 노트북 (10-phase1) 과 **동일한 hyperparameters** 를 사용해 직접 비교 가능하도록 한다. 차이는 시드만.

In [ ]:
ROOT = repo_root()
DATA_PATH = ROOT / "data" / "tinyshakespeare.txt"
OUT_DIR = ROOT / "runs" / "notebook-phase1-reproducibility"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEEDS = [42, 123, 456]  # 시드 invariance 검증용
DET_SEED = 42  # 결정론 검증용 (2회 반복)

BATCH_SIZE = 16
BLOCK_SIZE = 128

BACKBONE = dict(hidden_dim=256, n_heads=4, ffn_dim=1024, n_init_layers=4)
TRAIN_BASE = dict(
    lr=3e-4,
    max_steps=1500,
    max_layers=8,
    trigger_window=100,
    trigger_epsilon=0.08,
    trigger_cooldown=150,
    trigger_min_history=100,
    device=DEVICE,
)

print("SEEDS  =", SEEDS)
print("DET_SEED =", DET_SEED, "(× 2회)")
for k, v in {**BACKBONE, **TRAIN_BASE}.items():
    print(f"  {k:22s} = {v}")

## 3. 데이터 로드

토큰화는 시드 무관 — text → vocab 매핑은 deterministic. 시드는 batch sampling 에만 영향.

In [ ]:
text = load_tinyshakespeare_text(DATA_PATH)
tokenizer = CharTokenizer(text)
dataset = TinyShakespeareDataset(text, tokenizer)
print(f"vocab_size : {tokenizer.vocab_size}, n_tokens : {len(dataset):,}")

## 4. 시드별 학습 (`SEEDS` loop)

각 시드마다 model fresh init + data iterator 별도 시드 적용 + train 1500 step. GPU 에서 시드당 약 1-2분.

결과는 `results[seed] = TrainResult` 로 수집.

In [ ]:
cfg = BackboneConfig(vocab_size=tokenizer.vocab_size, max_seq_len=BLOCK_SIZE, **BACKBONE)
train_cfg = TrainConfig(**TRAIN_BASE)

results = {}
for seed in SEEDS:
    print(f"--- seed = {seed} ---")
    set_seed(seed)
    model = GrowingDecoder(cfg).to(DEVICE)
    data_iter = iter_random_batches(
        dataset, batch_size=BATCH_SIZE, block_size=BLOCK_SIZE, seed=seed
    )
    result = train(model, data_iter, train_cfg)
    results[seed] = result
    print(
        f"  done: n_layers={result.final_n_layers}, grow_events={len(result.grow_events)}, "
        f"final_loss≈{result.losses[-1]:.4f}"
    )

## 5. 시드별 결과 비교 (요약 표)

3 핵심 가설이 모든 시드에서 통과했는지, 시드 간 분산은 얼마나 되는지.

In [ ]:
import statistics

print(f"{'seed':>5}  {'n_layers':>9}  {'#grow':>6}  {'first':>9}  {'last':>9}  {'Δ':>8}")
print("-" * 60)
rows = []
for seed, r in results.items():
    # 분모를 실제 길이로 보정 — max_steps < 100 케이스 / losses 가 비는 케이스 방지
    n = min(100, len(r.losses))
    first = sum(r.losses[:n]) / n if n > 0 else 0.0
    last = sum(r.losses[-n:]) / n if n > 0 else 0.0
    rows.append((seed, r.final_n_layers, len(r.grow_events), first, last))
    print(
        f"{seed:>5}  {r.final_n_layers:>9}  {len(r.grow_events):>6}  {first:>9.4f}  {last:>9.4f}  {first - last:>+8.4f}"
    )
print()

# 시드 간 분산
n_layers_list = [r.final_n_layers for r in results.values()]
n_grow_list = [len(r.grow_events) for r in results.values()]
last_loss_list = []
for r in results.values():
    n = min(100, len(r.losses))
    last_loss_list.append(sum(r.losses[-n:]) / n if n > 0 else 0.0)
print("seed 간 분산:")
print(f"  final n_layers   : {n_layers_list} (모두 같으면 잘 수렴 = max_layers 도달)")
print(f"  #grow events     : {n_grow_list}")
print(
    f"  last avg loss    : mean={statistics.mean(last_loss_list):.4f}  σ={statistics.stdev(last_loss_list) if len(last_loss_list) > 1 else 0:.4f}"
)

## 6. 시각화 — 시드별 loss curve overlay

시드 간 학습 곡선의 형태가 유사해야 함 (시작점 / 수렴 속도 / 최종 수준).

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(11, 4))
for seed, r in results.items():
    ax.plot(range(1, len(r.losses) + 1), r.losses, lw=0.7, label=f"seed={seed}", alpha=0.75)
    for step, _n in r.grow_events:
        ax.axvline(step, color="gray", alpha=0.15, lw=0.6)
ax.set_xlabel("step")
ax.set_ylabel("CE loss")
ax.set_title("Phase 1 reproducibility — loss curves across seeds")
ax.legend(loc="upper right")
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(OUT_DIR / "curves_overlay.png", dpi=120)
plt.show()

## 7. 3 핵심 가설 시드별 통과 여부

각 시드 결과에 대해 Base 노트북과 동일한 3 가설 체크 (no spike 검증은 §8 의 spike 정량 검증과 별도).

In [ ]:
SPIKE_WINDOW = 30
SPIKE_THRESHOLD = 0.5  # Base 노트북 cell-14 와 동일 정의

print(f"{'seed':>5}  {'fire':>5}  {'no_spike':>9}  {'deeper':>7}  {'lower':>6}  ALL?")
print("-" * 55)
all_pass = True
for seed, r in results.items():
    fire = len(r.grow_events) >= 1
    deeper = r.final_n_layers > BACKBONE["n_init_layers"]
    # 분모 보정 — Base 노트북과 동일 패턴
    n = min(100, len(r.losses))
    first = sum(r.losses[:n]) / n if n > 0 else 0.0
    last = sum(r.losses[-n:]) / n if n > 0 else 0.0
    lower = last < first
    # no_spike: 각 grow event 전후 30 step 평균 비교 (Base 노트북 cell-14 와 동일 로직)
    no_spike = True
    evaluated = 0
    for step, _n in r.grow_events:
        pre = r.losses[max(0, step - SPIKE_WINDOW) : step]
        post = r.losses[step : step + SPIKE_WINDOW]
        if not pre or not post:
            continue
        delta = sum(post) / len(post) - sum(pre) / len(pre)
        if abs(delta) > SPIKE_THRESHOLD:
            no_spike = False
        evaluated += 1
    # evaluable event 없으면 검증 불가 — N/A 로 표시 (PASS 로 잘못 보고 차단)
    no_spike_ok = no_spike and evaluated > 0
    no_spike_str = "N/A" if evaluated == 0 else ("✓" if no_spike else "✗")

    ok = fire and no_spike_ok and deeper and lower
    all_pass &= ok
    print(
        f"{seed:>5}  {('✓' if fire else '✗'):>5}  {no_spike_str:>9}  {('✓' if deeper else '✗'):>7}  {('✓' if lower else '✗'):>6}  {'PASS' if ok else 'FAIL'}"
    )
print()
print("seed invariance:", "OK (모든 시드 통과)" if all_pass else "FAIL")

## 8. 결정론 검증 — 동일 시드 (`DET_SEED`) 2회 실행

GPU 의 cuBLAS / cuDNN non-deterministic 커널 때문에 비트 단위 동일을 기대하지 않고, `torch.allclose(rtol=1e-3)` 의 **근사 결정론** 을 본다. CPU 환경이면 비트 단위 동일이 기대됨.

In [ ]:
det_results = []
for run_idx in (0, 1):
    print(f"--- DET run {run_idx + 1} (seed={DET_SEED}) ---")
    set_seed(DET_SEED)
    model = GrowingDecoder(cfg).to(DEVICE)
    data_iter = iter_random_batches(
        dataset, batch_size=BATCH_SIZE, block_size=BLOCK_SIZE, seed=DET_SEED
    )
    r = train(model, data_iter, train_cfg)
    det_results.append(r)
    print(f"  done: grow_events={r.grow_events}")

r1, r2 = det_results
losses1 = torch.tensor(r1.losses)
losses2 = torch.tensor(r2.losses)
max_diff = (losses1 - losses2).abs().max().item()
events_equal = r1.grow_events == r2.grow_events
exact = torch.equal(losses1, losses2)
close_strict = torch.allclose(losses1, losses2, rtol=1e-5, atol=1e-6)
close_loose = torch.allclose(losses1, losses2, rtol=1e-3, atol=1e-3)

print()
print(f"grow_events 동일       : {events_equal}")
print(f"max(|Δloss|) 비교       : {max_diff:.2e}")
print(f"비트 단위 동일 (equal)  : {exact}")
print(f"엄격 근사 (rtol=1e-5)   : {close_strict}")
print(f"느슨 근사 (rtol=1e-3)   : {close_loose}")
print()
# 결정론 판정의 1차 게이트는 grow_events 동등성 — events 가 다르면 어떤 loss 근사도 결정론 X
if not events_equal:
    print("결정론: 비결정론 (grow_events 불일치 — trigger fire 시점이 다름)")
elif exact:
    print("결정론: 완전 결정론 (CPU 환경 또는 deterministic algorithms 활성)")
elif close_strict:
    print("결정론: 부동소수점 epsilon 차이만 — 사실상 결정론")
elif close_loose:
    print("결정론: 근사 결정론 (GPU cuBLAS non-determinism 추정 — 학습에는 영향 없음)")
else:
    print(
        "결정론: 차이가 큼 — set_seed / data_iter seed 외에 시드 영향 받는 경로가 있는지 점검 필요"
    )

## 결과 요약 / 다음 가설

이 노트북에서 검증된 것:
- (가설 1) **seed invariance**: §7 의 PASS 라인이 모든 시드에서 통과
- (가설 2) **결정론**: §8 의 근사 결정론 (CPU 면 비트 단위, GPU 면 rtol=1e-3 이내)
- (가설 3) **분산**: §5 의 last 100 avg σ < 0.1 정도 (시드 간 작은 분산)

**Phase 1 보완 다음 항목**:
- #3 Trigger 튜닝 — epsilon=0.02-0.04 로 보수화
- #4 α 분포 검토 — Base 노트북 cell-16 의 출력을 시드별로 비교